# Collaborative filtering
Student information here!!!

This notebook builds two collaborative-filtering recommenders on the MovieLens ratings:

1. **Dataset analysis**: what the ratings matrix looks like.
2. **Similarity measures**: cosine, Pearson and adjusted cosine, checked against the slide example.
3. **User-user recommendations**: predict a rating from the most similar users.
4. **Item-item recommendations**: predict a rating from the movies the user already liked.
5. **Validation**: hit rate on a hidden rating per user, compared with a popularity baseline.
6. **Recommendation widget**: pick a method, a similarity measure, a user and N.

In [26]:
from IPython.core.display import HTML
from IPython.display import display
from movie_display import movie_display
import ipywidgets as widgets
import pandas as pd
import numpy as np
from math import sqrt

## 1. Dataset analysis
Every rating is a `(userId, movieId, rating, timestamp)` row. Pivoting turns this into a user × movie matrix
where a missing value (NaN) means the user has not rated the movie.

In [ ]:
df = pd.read_csv("../data/movielens/ratings.csv")
movies = pd.read_csv("../data/movielens/movies.csv")
titles = movies.set_index("movieId")["title"]

user_item = df.pivot(index="userId", columns="movieId", values="rating")
print(df.head(5))

n_users, n_movies = user_item.shape
raters_per_movie = df.groupby("movieId").size()

print(f"\n{len(df)} ratings, {n_users} users, {n_movies} movies")
print(f"Matrix filled: {len(df) / (n_users * n_movies):.2%}")
print(f"Ratings per user:  min {df.groupby('userId').size().min()}, median {df.groupby('userId').size().median():.0f}")
print(f"Ratings per movie: median {raters_per_movie.median():.0f}, "
      f"{(raters_per_movie == 1).sum()} movies rated only once, {(raters_per_movie >= 30).sum()} with >= 30 ratings")

   userId  movieId  rating   timestamp
0       1       31     2.5  1260759144
1       1     1029     3.0  1260759179
2       1     1061     3.0  1260759182
3       1     1129     2.0  1260759185
4       1     1172     4.0  1260759205

100004 ratings, 671 users, 9066 movies
Matrix filled: 1.64%
Ratings per user:  min 20, median 71
Ratings per movie: median 3, 3063 movies rated only once, 848 with >= 30 ratings


The matrix is very sparse (about 1.6% filled) and most movies have only a handful of ratings.
A prediction based on one or two ratings is unreliable, so both recommenders get a `min_raters=30`
parameter that only recommends movies with at least 30 ratings.

## 2. Similarity measures
All three measures are cosine similarities on (possibly shifted) rating vectors, with missing ratings treated as 0:

* **Cosine**: the raw ratings.
* **Pearson**: each vector minus its own mean, which removes "this user rates everything high".
* **Adjusted cosine**: each value minus the mean of the *other* dimension (for users: the movie's mean).

The slide example has users A–D and the movies HP1, HP2, HP3, TW, SW1, SW2, SW3.

In [28]:
def cosineSimilarity(object1,object2):
        object1 = np.nan_to_num(object1)
        object2 = np.nan_to_num(object2)
        # sum_object1 = 0
        # sum_object2 = 0
        # for i in range(len(object1)):
        #     sum_object1 += object1[i]**2
        #     sum_object2 += object2[i]**2

        # denominator = sqrt(sum_object1) * sqrt(sum_object2)
        denominator = np.linalg.norm(object1) * np.linalg.norm(object2)
        cosSim= object1 @ object2 / denominator if denominator else 0.0
        return cosSim

def pearsonSim(a,b):
    # only use positions where BOTH rated, centred on their own mean
    # both = ~np.isnan(a) & ~np.isnan(b)
    
    # if both.sum() < 2:
    #     return 0.0
    
    a = np.nan_to_num(a - np.nanmean(a))
    b = np.nan_to_num(b - np.nanmean(b))
    
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return a @ b / denom if denom else 0.0

def adjustedCosSim(a, b, item_means):
    a = np.nan_to_num(a - item_means)
    b = np.nan_to_num(b - item_means)

    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return a @ b / denom if denom else 0.0

# slide example: users A-D, movies HP1 HP2 HP3 TW SW1 SW2 SW3
nan = np.nan
slide = np.array([[4, nan, nan, 5,   1,   nan, nan],
                  [5, 5,   4,   nan, nan, nan, nan],
                  [nan, nan, nan, 2,   4,   5,   nan],
                  [nan, 3,   nan, nan, nan, nan, 3]])

print("Cosine  A,B:", cosineSimilarity(slide[0], slide[1]))   # slides: 0.38
print("Pearson A,B:", pearsonSim(slide[0], slide[1]))         # slides: 0.092
item_means = np.nanmean(slide, axis=0)                        # one mean per movie
print("Adj cos A,B:", adjustedCosSim(slide[0], slide[1], item_means))  # slides: -0.1025

Cosine  A,B: 0.3798685881987931
Pearson A,B: 0.09245003270420475
Adj cos A,B: -0.1025978352085154


`similarity_matrix` computes the same measure for every pair of rows at once: normalise the rows, then one
matrix product. Rows are users for user-user and movies (`R.T`) for item-item. The check below makes sure
the matrix version gives the slide numbers for users A and B (rows 0 and 1).

In [29]:
def similarity_matrix(M, metric="cosine"):
    """Similarity between every pair of ROWS of M (NaN = not rated)."""
    M = M.astype(np.float32)
   
    if metric == "pearson":                    # subtract each vector's own mean
        M = M - np.nanmean(M, axis=1, keepdims=True)
    elif metric == "adjusted cosine":          # subtract the mean of the other dimension
        M = M - np.nanmean(M, axis=0, keepdims=True)
    
    X = np.nan_to_num(M)                       # missing -> 0
    
    norms = np.linalg.norm(X, axis=1)
    norms[norms == 0] = 1                      # zero vector -> similarity 0
    
    X = X / norms[:, None]
    S = X @ X.T                                # all pairs at once
    np.fill_diagonal(S, 1)
    
    return S

slide_values = {"cosine": 0.38, "pearson": 0.092, "adjusted cosine": -0.1025}
for metric, expected in slide_values.items():
    got = similarity_matrix(slide, metric)[0, 1]
    print(f"{metric:16s} A,B = {got:.4f}   slides: {expected}")
    assert np.isclose(got, expected, atol=1e-3), metric

cosine           A,B = 0.3799   slides: 0.38
pearson          A,B = 0.0925   slides: 0.092
adjusted cosine  A,B = -0.1026   slides: -0.1025


For the real data the user similarity matrices (671 × 671) are built up front. The item matrices are
9066 × 9066 (~330 MB each), so they are only computed the first time a measure is used.

In [30]:
R = user_item.to_numpy(dtype=np.float32)       # rows = users, columns = movies
user_sims = {m: similarity_matrix(R, m) for m in ["cosine", "pearson", "adjusted cosine"]}  # 671x671
item_sims = {}                                  # filled on demand: each is 9066x9066, ~330 MB

def get_item_sims(metric):
    if metric not in item_sims:
        item_sims[metric] = similarity_matrix(R.T, metric)
    return item_sims[metric]

## 3. User-user recommendations
For user *u*:

1. Take the *k* most similar users with a positive similarity (the neighbours).
2. For each movie *u* has not seen, the prediction is the similarity-weighted average of the neighbours' ratings:
   $\hat r_{u,i} = \frac{\sum_v s_{u,v}\, r_{v,i}}{\sum_v s_{u,v}}$ (sum over the neighbours *v* that rated *i*).
3. Only movies with at least `min_raters` ratings in the whole dataset can be recommended.

A prediction from a single neighbour is as "sure" as one from all ten, so ranking by the prediction
alone favours obscure movies that one neighbour loved. The ranking therefore uses
$\text{score} = \hat r \cdot \frac{\text{count}}{\text{count} + 1}$, where *count* is the number of neighbours
that rated the movie. The function still returns the real prediction $\hat r$.

In [31]:
def recommend_user_user(R, S_user, u, n=10, k=10, min_raters=30):
    sims = S_user[u].copy();
    sims[u] = -np.inf

    nb = np.argsort(-sims)[:k]                 # the k most similar users
    nb = nb[sims[nb] > 0]                      # only positively similar ones

    w = sims[nb]
    r = R[nb]
    rated = ~np.isnan(r)
    num = (np.nan_to_num(r) * w[:, None]).sum(axis=0)
    den = (rated * w[:, None]).sum(axis=0)
    count = rated.sum(axis=0)                  # how many neighbours rated each movie
    popular = (~np.isnan(R)).sum(axis=0) >= min_raters   # enough ratings in the whole dataset
    ok = np.isnan(R[u]) & (den > 0) & popular  # unseen by u, rated by >= 1 neighbour
    pred = np.full(R.shape[1], np.nan); pred[ok] = num[ok] / den[ok]
    score = pred * count / (count + 1)         # shrink predictions backed by few neighbours
    cand = np.where(ok)[0]
    order = cand[np.lexsort((-count[cand], -score[cand]))][:n]

    return order, pred[order]

### Toy example
Four users, four movies, cosine similarity. U0 has seen A and B; which of C and D comes first?

In [32]:
toy = np.array([[5,   4, nan, nan],
                [4,   3, 4,   5  ],
                [5,   2, 5,   nan],
                [nan, 1, 2,   1  ]])
toy_users, toy_movies = ["U0", "U1", "U2", "U3"], ["A", "B", "C", "D"]
display(pd.DataFrame(toy, index=toy_users, columns=toy_movies))

S_toy = similarity_matrix(toy, "cosine")
print("cosine similarity of U0 with U1..U3:", S_toy[0, 1:].round(3))

order, pred = recommend_user_user(toy, S_toy, 0, n=2, k=2, min_raters=1)
for i, p in zip(order, pred):
    print(f"{toy_movies[i]}: predicted {p:.2f}")

,A,B,C,D
U0,5.0,4.0,NaN,NaN
U1,4.0,3.0,4.0,5.0
U2,5.0,2.0,5.0,NaN
U3,NaN,1.0,2.0,1.0


cosine similarity of U0 with U1..U3: [0.615 0.701 0.255]
C: predicted 4.53
D: predicted 5.00


**By hand** (missing ratings count as 0 in the cosine):

* $\cos(U0, U1) = \frac{5\cdot4 + 4\cdot3}{\sqrt{41}\,\sqrt{66}} = \frac{32}{52.02} = 0.615$
* $\cos(U0, U2) = \frac{5\cdot5 + 4\cdot2}{\sqrt{41}\,\sqrt{54}} = \frac{33}{47.05} = 0.701$
* $\cos(U0, U3) = \frac{4\cdot1}{\sqrt{41}\,\sqrt{6}} = \frac{4}{15.68} = 0.255$

With $k=2$ the neighbours are U2 and U1.

| movie | neighbours who rated it | prediction | count | score = pred · count/(count+1) |
|---|---|---|---|---|
| C | U2 (5), U1 (4) | $\frac{0.701\cdot5 + 0.615\cdot4}{0.701+0.615} = 4.53$ | 2 | $4.53 \cdot 2/3 = 3.02$ |
| D | U1 (5) | $\frac{0.615\cdot5}{0.615} = 5.00$ | 1 | $5.00 \cdot 1/2 = 2.50$ |

D has the higher prediction, but only one neighbour rated it. C is backed by both neighbours, so it is
recommended first. The output is C (4.53), then D (5.00).

## 4. Item-item recommendations
For user *u*:

1. Take the movies *u* rated at least `min_rating` (3.5): the movies *u* liked.
2. For every candidate movie *i*, keep its *j* most similar liked movies (negative similarities count as 0).
3. The prediction is the similarity-weighted average of *u*'s own ratings of those liked movies:
   $\hat r_{u,i} = \frac{\sum_l s_{i,l}\, r_{u,l}}{\sum_l s_{i,l}}$.
4. As before, only movies with at least `min_raters` ratings can be recommended.

In [33]:
def recommend_item_item(R, S_item, u, n=10, min_rating=3.5, j=10, min_raters=30):
    liked = np.where(R[u] >= min_rating)[0]    # movies u rated >= 3.5

    if len(liked) == 0:
        return np.array([], int), np.array([])

    sub = S_item[liked].T.copy()               # every movie vs the liked ones (S is symmetric; rows slice faster)
    sub[sub < 0] = 0

    if len(liked) > j:                         # keep the j most similar liked movies per candidate
        thresh = -np.partition(-sub, j - 1, axis=1)[:, j - 1:j]
        sub[sub < thresh] = 0

    num, den = sub @ R[u, liked], sub.sum(axis=1)
    popular = (~np.isnan(R)).sum(axis=0) >= min_raters   # enough ratings in the whole dataset
    ok = np.isnan(R[u]) & (den > 0) & popular
    pred = np.full(R.shape[1], np.nan); pred[ok] = num[ok] / den[ok]
    cand = np.where(ok)[0]
    order = cand[np.lexsort((-den[cand], -pred[cand]))][:n]

    return order, pred[order]

### Toy example
The same toy matrix, now comparing movies (the columns) instead of users.

In [34]:
S_toy_items = similarity_matrix(toy.T, "cosine")
display(pd.DataFrame(S_toy_items, index=toy_movies, columns=toy_movies).round(3))

order, pred = recommend_item_item(toy, S_toy_items, 0, n=2, min_raters=1)
for i, p in zip(order, pred):
    print(f"{toy_movies[i]}: predicted {p:.2f}")

,A,B,C,D
A,1.000,0.944,0.752,0.483
B,0.944,1.000,0.653,0.573
C,0.752,0.653,1.000,0.643
D,0.483,0.573,0.643,1.000


C: predicted 4.54
D: predicted 4.46


**By hand.** U0 liked A (5) and B (4). The movie vectors over U0..U3 are
A = (5, 4, 5, 0), B = (4, 3, 2, 1), C = (0, 4, 5, 2), D = (0, 5, 0, 1).

* $\cos(C, A) = \frac{4\cdot4 + 5\cdot5}{\sqrt{45}\,\sqrt{66}} = \frac{41}{54.50} = 0.752$,
  $\cos(C, B) = \frac{4\cdot3 + 5\cdot2 + 2\cdot1}{\sqrt{45}\,\sqrt{30}} = \frac{24}{36.74} = 0.653$
* $\cos(D, A) = \frac{5\cdot4}{\sqrt{26}\,\sqrt{66}} = \frac{20}{41.42} = 0.483$,
  $\cos(D, B) = \frac{5\cdot3 + 1\cdot1}{\sqrt{26}\,\sqrt{30}} = \frac{16}{27.93} = 0.573$

Predictions from U0's own ratings:

* C: $\frac{0.752\cdot5 + 0.653\cdot4}{0.752 + 0.653} = \frac{6.37}{1.405} = 4.54$
* D: $\frac{0.483\cdot5 + 0.573\cdot4}{0.483 + 0.573} = \frac{4.71}{1.056} = 4.46$

C is more similar to the movie U0 rated highest (A), so it comes first: C (4.54), then D (4.46).

## From row positions to userId and movieId
The recommenders work with row and column positions in `R`. `user_item.index.get_loc` turns a `userId` into
its row, and `user_item.columns[order]` turns the returned column positions back into `movieId`s.

In [35]:
def recommend(user_id, method="user-user", metric="cosine", n=10, **kwargs):
    """Top-n recommendations for a userId as a DataFrame (movieId, title, predicted rating)."""
    u = user_item.index.get_loc(user_id)       # userId -> row position
    if method == "user-user":
        order, pred = recommend_user_user(R, user_sims[metric], u, n, **kwargs)
    else:
        order, pred = recommend_item_item(R, get_item_sims(metric), u, n, **kwargs)
    movie_ids = user_item.columns[order]       # column positions -> movieId
    return pd.DataFrame({"movieId": movie_ids,
                         "title": titles.reindex(movie_ids).values,
                         "predicted": pred.round(2)})

display(recommend(1, "user-user", "pearson"))
display(recommend(1, "item-item", "cosine"))

,movieId,title,predicted
0,1196,Star Wars: Episode V - The Empire Strikes Back...,4.70
1,2571,"Matrix, The (1999)",4.86
2,904,Rear Window (1954),4.56
3,260,Star Wars: Episode IV - A New Hope (1977),4.35
4,296,Pulp Fiction (1994),4.50
5,1376,Star Trek IV: The Voyage Home (1986),4.65
6,3072,Moonstruck (1987),4.56
7,1393,Jerry Maguire (1996),4.52
8,780,Independence Day (a.k.a. ID4) (1996),4.20
9,1210,Star Wars: Episode VI - Return of the Jedi (1983),4.48


,movieId,title,predicted
0,8533,"Notebook, The (2004)",3.97
1,55247,Into the Wild (2007),3.96
2,8984,Ocean's Twelve (2004),3.95
3,4235,Amores Perros (Love's a Bitch) (2000),3.95
4,720,Wallace & Gromit: The Best of Aardman Animatio...,3.95
5,2324,Life Is Beautiful (La Vita è bella) (1997),3.94
6,2948,From Russia with Love (1963),3.94
7,6218,Bend It Like Beckham (2002),3.94
8,3435,Double Indemnity (1944),3.93
9,1183,"English Patient, The (1996)",3.93


## 5. Validation: hit rate
For every user with at least 10 ratings, hide their most recent rating of 3.5 or higher, a movie we know
they liked. The similarity matrices are rebuilt on the remaining ratings, so the hidden rating cannot leak into
them. A **hit** means the hidden movie appears in the user's top N; the hit rate is the fraction of users with a hit.

The **popularity baseline** recommends the most-rated movies the user has not seen yet. A recommender is only
useful if it beats that.

Note: a hidden movie with fewer than 30 ratings can never be a hit because of `min_raters`, so the hit rates
are a lower bound.

In [36]:
counts = df.groupby("userId")["rating"].transform("size")
liked = df[(counts >= 10) & (df["rating"] >= 3.5)]
hidden = liked.sort_values("timestamp", kind="stable").groupby("userId").tail(1)

hid_u = user_item.index.get_indexer(hidden["userId"])     # userId -> row
hid_m = user_item.columns.get_indexer(hidden["movieId"])  # movieId -> column
R_train = R.copy()
R_train[hid_u, hid_m] = np.nan
print(f"{len(hidden)} users, one hidden rating each")

Ns = [5, 10, 20]

def hit_rates(recommend_fn):
    """recommend_fn(u, N) -> column positions; returns the hit rate for every N in Ns."""
    tops = [recommend_fn(u, max(Ns)) for u in hid_u]
    return {f"hit@{N}": np.mean([m in top[:N] for top, m in zip(tops, hid_m)]) for N in Ns}

by_popularity = np.argsort(-(~np.isnan(R_train)).sum(axis=0), kind="stable")

def recommend_popular(u, N):
    return by_popularity[np.isnan(R_train[u, by_popularity])][:N]

results = {("popularity", "-"): hit_rates(recommend_popular)}
for metric in ["cosine", "pearson", "adjusted cosine"]:
    S = similarity_matrix(R_train, metric)
    results[("user-user", metric)] = hit_rates(lambda u, N: recommend_user_user(R_train, S, u, N)[0])
    S = similarity_matrix(R_train.T, metric)   # one 9066x9066 matrix at a time
    results[("item-item", metric)] = hit_rates(lambda u, N: recommend_item_item(R_train, S, u, N)[0])
    del S

hit_table = (pd.DataFrame(results).T * 100).round(1)
hit_table.columns = [f"{c} (%)" for c in hit_table.columns]
hit_table

671 users, one hidden rating each


C:\Users\ivana\AppData\Local\Temp\ipykernel_6696\717629110.py:6: RuntimeWarning: Mean of empty slice
  M = M - np.nanmean(M, axis=1, keepdims=True)
C:\Users\ivana\AppData\Local\Temp\ipykernel_6696\717629110.py:8: RuntimeWarning: Mean of empty slice
  M = M - np.nanmean(M, axis=0, keepdims=True)


,,hit@5 (%),hit@10 (%),hit@20 (%)
popularity,-,3.3,5.2,7.5
user-user,cosine,5.7,9.1,13.1
item-item,cosine,1.0,1.5,2.4
user-user,pearson,3.7,6.1,11.3
item-item,pearson,0.9,1.5,2.1
user-user,adjusted cosine,4.0,7.5,11.0
item-item,adjusted cosine,0.3,0.7,1.6


**Results.** User-user with cosine similarity is the best (about 9% at N=10) and clearly beats the popularity
baseline (about 5%). Pearson and adjusted cosine are in between. Item-item scores below the baseline: its prediction
is an average of the user's *own* ratings of liked movies (all ≥ 3.5), so almost every candidate gets a prediction
close to 4. The ranking then separates the candidates poorly and favours movies that resemble just a few liked ones.

## 6. Recommendation widget
Pick the method, the similarity measure, a user and N. Movie details come from joining `links.csv`
(movieId → imdbId) to `imdbdata.json` on `imdbId`; the predicted rating (out of 5) is shown above each plot.

In [40]:
links = pd.read_csv("../data/movielens/links.csv", dtype={"imdbId": str})
imdb = pd.read_json("../data/movielens/imdbdata.json", dtype={"imdbId": str})

movie_info = links.merge(imdb, on="imdbId", how="left").set_index("movieId")
movie_info["Title"] = movie_info["Title"].fillna(titles)   # a few movies have no IMDB entry
movie_info = movie_info.fillna("").replace("N/A", "")
movie_info.loc[movie_info["imdbRating"] == "", "imdbRating"] = "0"
movie_info["imdbId"] = movie_info["imdbId"].str[1:]         # movie_display adds the "tt0" prefix itself

def show_recommendations(method, metric, user_id, n):
    recs = recommend(user_id, method, metric, n)
    if recs.empty:
        print("No recommendations for this user.")
        return
    items = []
    for movie_id, pred in zip(recs["movieId"], recs["predicted"]):
        item = movie_info.loc[movie_id].copy()
        item["Plot"] = f"<b>Predicted rating: {pred:.2f} / 5</b><br>{item['Plot']}"
        items.append(item)
    display(HTML(movie_display.show(items)))

widgets.interact(
    show_recommendations,
    method=widgets.ToggleButtons(options=["user-user", "item-item"], description="Method"),
    metric=widgets.Dropdown(options=["cosine", "pearson", "adjusted cosine"], description="Similarity"),
    user_id=widgets.Dropdown(options=list(user_item.index), description="userId"),
    n=widgets.IntSlider(value=10, min=1, max=30, description="N", continuous_update=False),
);

interactive(children=(ToggleButtons(description='Method', options=('user-user', 'item-item'), value='user-user…